In [3]:
import requests
import pandas as pd
import os
import json
from dotenv import load_dotenv
import matplotlib.pyplot as plt

In [4]:
load_dotenv()
EPI_API_KEY = os.getenv("EPI_API_KEY")

In [5]:
# API call to database

meta_url = "https://api.eia.gov/v2/electricity/electric-power-operational-data"
params = {"api_key": EPI_API_KEY}

r = requests.get(meta_url, params=params) # 'get' visits the website, returns JSON
print(r.status_code) # printing the status code, 200 = success

meta = r.json() 

print(json.dumps(meta, indent=2)[:4000]) # prints the json so it is readable

200
{
  "response": {
    "id": "electric-power-operational-data",
    "name": "Electric Power Operations (Annual and Monthly)",
    "description": "Monthly and annual electric power operations by state, sector, and energy source.\n    Source: Form EIA-923",
    "frequency": [
      {
        "id": "monthly",
        "description": "One data point for each month.",
        "query": "M",
        "format": "YYYY-MM"
      },
      {
        "id": "quarterly",
        "description": "One data point every 3 months.",
        "query": "Q",
        "format": "YYYY-\"Q\"Q"
      },
      {
        "id": "annual",
        "description": "One data point for each calendar year.",
        "query": "A",
        "format": "YYYY"
      }
    ],
    "facets": [
      {
        "id": "location",
        "description": "State / Census Region"
      },
      {
        "id": "sectorid",
        "description": "Sector"
      },
      {
        "id": "fueltypeid",
        "description": "Energy Source"
   

In [6]:
for f in meta["response"]["frequency"]:
    print(f["id"], f["query"])

monthly M
quarterly Q
annual A


In [7]:
# pull real data
meta_url1 = "https://api.eia.gov/v2/electricity/electric-power-operational-data/data/"

# coal, natural gas, nuclear, hydro, wind, solar.
params1 = {'frequency':'annual',
           'data[]':'generation',
           'facets[location][]':'US',
            'facets[sectorid][]':'99',
            'facets[fueltypeid][]':['COW','NG','NUC','SUN','WND','HYC'],
           'start':'2005',
           'end':'2025',
           'sort[0][column]':'period',
            'sort[0][direction]':'asc',
            'api_key': EPI_API_KEY
           }

r = requests.get(meta_url1, params=params1) # 'get' visits the website, returns JSON
print(r.status_code) # printing the status code, 200 = success

meta = r.json() 



200


In [8]:
print("Total records:", meta["response"]["total"])
print(json.dumps(meta["response"]["data"][:3], indent=2))

Total records: 126
[
  {
    "period": "2005",
    "location": "US",
    "stateDescription": "U.S. Total",
    "sectorid": "99",
    "sectorDescription": "All Sectors",
    "fueltypeid": "COW",
    "fuelTypeDescription": "all coal products",
    "generation": "2012873.04581",
    "generation-units": "thousand megawatthours"
  },
  {
    "period": "2005",
    "location": "US",
    "stateDescription": "U.S. Total",
    "sectorid": "99",
    "sectorDescription": "All Sectors",
    "fueltypeid": "HYC",
    "fuelTypeDescription": "conventional hydroelectric",
    "generation": "270321.25494",
    "generation-units": "thousand megawatthours"
  },
  {
    "period": "2005",
    "location": "US",
    "stateDescription": "U.S. Total",
    "sectorid": "99",
    "sectorDescription": "All Sectors",
    "fueltypeid": "NG",
    "fuelTypeDescription": "natural gas",
    "generation": "760960.25439",
    "generation-units": "thousand megawatthours"
  }
]


In [9]:
eia_df = pd.DataFrame(meta["response"]["data"])
eia_df[eia_df['sectorid'] == '99']

,period,location,stateDescription,sectorid,sectorDescription,fueltypeid,fuelTypeDescription,generation,generation-units
0,2005,US,U.S. Total,99,All Sectors,COW,all coal products,2012873.04581,thousand megawatthours
1,2005,US,U.S. Total,99,All Sectors,HYC,conventional hydroelectric,270321.25494,thousand megawatthours
2,2005,US,U.S. Total,99,All Sectors,NG,natural gas,760960.25439,thousand megawatthours
3,2005,US,U.S. Total,99,All Sectors,NUC,nuclear,781986.365,thousand megawatthours
4,2005,US,U.S. Total,99,All Sectors,SUN,solar,550.29434,thousand megawatthours
...,...,...,...,...,...,...,...,...,...
121,2025,US,U.S. Total,99,All Sectors,HYC,conventional hydroelectric,247023.20955,thousand megawatthours
122,2025,US,U.S. Total,99,All Sectors,NG,natural gas,1807338.28797,thousand megawatthours
123,2025,US,U.S. Total,99,All Sectors,NUC,nuclear,784780.718,thousand megawatthours
124,2025,US,U.S. Total,99,All Sectors,SUN,solar,295671.38248,thousand megawatthours


In [10]:
# seeing the data
print(f'Periods:{eia_df['period'].unique()}')
print(f'Locations: {eia_df['location'].unique()}')
print(f'State Descriptions: {eia_df['stateDescription'].unique()}')
print(f'Sector ID:  {eia_df['sectorid'].unique()}')
print(f'Sector Descriptions:  {eia_df['sectorDescription'].unique()}')
print(f'Fuel Type ID:  {eia_df['fueltypeid'].unique()}')
print(f'Fuel Type Description:  {eia_df['fuelTypeDescription'].unique()}')


Periods:['2005' '2006' '2007' '2008' '2009' '2010' '2011' '2012' '2013' '2014'
 '2015' '2016' '2017' '2018' '2019' '2020' '2021' '2022' '2023' '2024'
 '2025']
Locations: ['US']
State Descriptions: ['U.S. Total']
Sector ID:  ['99']
Sector Descriptions:  ['All Sectors']
Fuel Type ID:  ['COW' 'HYC' 'NG' 'NUC' 'SUN' 'WND']
Fuel Type Description:  ['all coal products' 'conventional hydroelectric' 'natural gas' 'nuclear'
 'solar' 'wind']


In [11]:
print("Total available:", meta["response"]["total"])
print("Rows returned:", len(eia_df))

Total available: 126
Rows returned: 126


In [12]:
eia_df.dtypes

period                 object
location               object
stateDescription       object
sectorid               object
sectorDescription      object
fueltypeid             object
fuelTypeDescription    object
generation             object
generation-units       object
dtype: object

In [13]:
# converting data
eia_df['period'] = eia_df['period'].astype(int)
eia_df['generation'] = eia_df['generation'].astype(float)

eia_df.head(20)


,period,location,stateDescription,sectorid,sectorDescription,fueltypeid,fuelTypeDescription,generation,generation-units
0,2005,US,U.S. Total,99,All Sectors,COW,all coal products,2.012873e+06,thousand megawatthours
1,2005,US,U.S. Total,99,All Sectors,HYC,conventional hydroelectric,2.703213e+05,thousand megawatthours
2,2005,US,U.S. Total,99,All Sectors,NG,natural gas,7.609603e+05,thousand megawatthours
3,2005,US,U.S. Total,99,All Sectors,NUC,nuclear,7.819864e+05,thousand megawatthours
4,2005,US,U.S. Total,99,All Sectors,SUN,solar,5.502943e+02,thousand megawatthours
5,2005,US,U.S. Total,99,All Sectors,WND,wind,1.781055e+04,thousand megawatthours
6,2006,US,U.S. Total,99,All Sectors,COW,all coal products,1.990511e+06,thousand megawatthours
7,2006,US,U.S. Total,99,All Sectors,HYC,conventional hydroelectric,2.892464e+05,thousand megawatthours
8,2006,US,U.S. Total,99,All Sectors,NG,natural gas,8.164408e+05,thousand megawatthours
9,2006,US,U.S. Total,99,All Sectors,NUC,nuclear,7.872186e+05,thousand megawatthours


In [14]:
filename1 = 'energy_mix.csv'
path1 = '/Users/colleenpangilinan/Documents/Colleen_Projects/us-emissions-dashboard:/data'
full_path = os.path.join(path1,filename1)
eia_df.to_csv(full_path,index=False)

In [15]:
# generation
eia_generation = eia_df.groupby('period')['generation'].sum().reset_index()
eia_generation.head(20)
# print(len(eia_generation))

,period,generation
0,2005,3.844502e+06
1,2006,3.910514e+06
2,2007,4.002042e+06
3,2008,3.986049e+06
4,2009,3.823960e+06
5,2010,3.998023e+06
6,2011,3.978672e+06
7,2012,3.930657e+06
8,2013,3.940407e+06
9,2014,3.964224e+06


In [16]:
# filename2 = 'carbon_intensity.csv'
# path2 = '/Users/colleenpangilinan/Documents/Colleen_Projects/us-emissions-dashboard:/data'
# full_path2 = os.path.join(path2,filename2)
# eia_carbon_intensity_df.to_csv(full_path2)

In [17]:
# API call to database

# pull real data
meta_url1 = "https://api.eia.gov/v2/co2-emissions/co2-emissions-aggregates/data/"

# coal, natural gas, nuclear, hydro, wind, solar.
params1 = {
    "frequency": "annual",
    "data[]":"value",
    'facets[sectorId][]':'EC',
    'facets[stateId][]':'US',
    'facets[fuelId][]':'TO',
    "start": '2005',
    "end": None,
    'sort[0][column]':'period',
    'sort[0][direction]':'asc',
    "offset": 0,
    "length": 5000,
    "api_key":EPI_API_KEY
    
}


r = requests.get(meta_url1, params=params1) # 'get' visits the website, returns JSON
print(r.status_code) # printing the status code, 200 = success

meta = r.json() 
print(r.text)
print()


200
{"warnings":[{"warning":"deprecated route","description":"This route is deprecated and no longer receiving fresh data. For state-level CO2 emissions data, please use the \/v2\/seds\/ API route."}],"response":{"total":"18","dateFormat":"YYYY","frequency":"annual","data":[{"period":"2005","sectorId":"EC","sector-name":"Electric Power carbon dioxide emissions","fuelId":"TO","fuel-name":"All Fuels","stateId":"US","state-name":"United States","value":"2400.544484","value-units":"million metric tons of CO2"},{"period":"2006","sectorId":"EC","sector-name":"Electric Power carbon dioxide emissions","fuelId":"TO","fuel-name":"All Fuels","stateId":"US","state-name":"United States","value":"2344.749681","value-units":"million metric tons of CO2"},{"period":"2007","sectorId":"EC","sector-name":"Electric Power carbon dioxide emissions","fuelId":"TO","fuel-name":"All Fuels","stateId":"US","state-name":"United States","value":"2410.99822","value-units":"million metric tons of CO2"},{"period":"2008

In [18]:
print("Total records:", meta["response"]["total"])
print(json.dumps(meta["response"]["data"][:3], indent=2))

Total records: 18
[
  {
    "period": "2005",
    "sectorId": "EC",
    "sector-name": "Electric Power carbon dioxide emissions",
    "fuelId": "TO",
    "fuel-name": "All Fuels",
    "stateId": "US",
    "state-name": "United States",
    "value": "2400.544484",
    "value-units": "million metric tons of CO2"
  },
  {
    "period": "2006",
    "sectorId": "EC",
    "sector-name": "Electric Power carbon dioxide emissions",
    "fuelId": "TO",
    "fuel-name": "All Fuels",
    "stateId": "US",
    "state-name": "United States",
    "value": "2344.749681",
    "value-units": "million metric tons of CO2"
  },
  {
    "period": "2007",
    "sectorId": "EC",
    "sector-name": "Electric Power carbon dioxide emissions",
    "fuelId": "TO",
    "fuel-name": "All Fuels",
    "stateId": "US",
    "state-name": "United States",
    "value": "2410.99822",
    "value-units": "million metric tons of CO2"
  }
]


In [24]:
emissions_df = pd.DataFrame(meta["response"]["data"])
print(emissions_df.columns.tolist())
emissions_df


['period', 'sectorId', 'sector-name', 'fuelId', 'fuel-name', 'stateId', 'state-name', 'value', 'value-units']


,period,sectorId,sector-name,fuelId,fuel-name,stateId,state-name,value,value-units
0,2005,EC,Electric Power carbon dioxide emissions,TO,All Fuels,US,United States,2400.544484,million metric tons of CO2
1,2006,EC,Electric Power carbon dioxide emissions,TO,All Fuels,US,United States,2344.749681,million metric tons of CO2
2,2007,EC,Electric Power carbon dioxide emissions,TO,All Fuels,US,United States,2410.99822,million metric tons of CO2
3,2008,EC,Electric Power carbon dioxide emissions,TO,All Fuels,US,United States,2359.66741,million metric tons of CO2
4,2009,EC,Electric Power carbon dioxide emissions,TO,All Fuels,US,United States,2145.715455,million metric tons of CO2
5,2010,EC,Electric Power carbon dioxide emissions,TO,All Fuels,US,United States,2259.628612,million metric tons of CO2
6,2011,EC,Electric Power carbon dioxide emissions,TO,All Fuels,US,United States,2158.915646,million metric tons of CO2
7,2012,EC,Electric Power carbon dioxide emissions,TO,All Fuels,US,United States,2023.916994,million metric tons of CO2
8,2013,EC,Electric Power carbon dioxide emissions,TO,All Fuels,US,United States,2038.751469,million metric tons of CO2
9,2014,EC,Electric Power carbon dioxide emissions,TO,All Fuels,US,United States,2037.785703,million metric tons of CO2


In [25]:
emissions_df = emissions_df[['period','value']].copy()
emissions_df.rename(columns={'value':'emissions'},inplace=True)
emissions_df

,period,emissions
0,2005,2400.544484
1,2006,2344.749681
2,2007,2410.99822
3,2008,2359.66741
4,2009,2145.715455
5,2010,2259.628612
6,2011,2158.915646
7,2012,2023.916994
8,2013,2038.751469
9,2014,2037.785703


In [26]:
eia_generation = eia_generation[(eia_generation['period'] > 2004) & (eia_generation['period'] <= 2022)]
emissions_df['period'] = emissions_df['period'].astype(int)
emissions_df['emissions'] = emissions_df['emissions'].astype(float)

print(eia_generation.dtypes)
print(emissions_df.dtypes)

period          int64
generation    float64
dtype: object
period         int64
emissions    float64
dtype: object


In [27]:
# emissions is in million metric tons of co2 -> hundred of g
# generation is in thousand mwh -> kwh
# emissions_df['emissions'] = emissions_df['emissions'] * 10000000000
# eia_generation['generation'] = eia_generation['generation'] * 1000


intensity_df = eia_generation.merge(emissions_df,on='period')
intensity_df['carbon_intensity'] = intensity_df['emissions']/intensity_df['generation'] * 1000000
intensity_df=intensity_df.rename(columns={'generation':'total_generation','emissions':'c02_emissions'})
intensity_df

,period,total_generation,c02_emissions,carbon_intensity
0,2005,3.844502e+06,2400.544484,624.409776
1,2006,3.910514e+06,2344.749681,599.601434
2,2007,4.002042e+06,2410.998220,602.442035
3,2008,3.986049e+06,2359.667410,591.981524
4,2009,3.823960e+06,2145.715455,561.123939
5,2010,3.998023e+06,2259.628612,565.186453
6,2011,3.978672e+06,2158.915646,542.622105
7,2012,3.930657e+06,2023.916994,514.905525
8,2013,3.940407e+06,2038.751469,517.396099
9,2014,3.964224e+06,2037.785703,514.043986
